# Day 2 EDA, Preprocessing, and Smoke-Test Notes

This notebook documents the Day 2 artifacts used before encoder training. Day 2 focuses on inspecting prepared data, tokenizing it into a HuggingFace `DatasetDict`, validating label alignment, and confirming the encoder pipeline with a small smoke test.


## Related Day 2 Files

| File | Role |
|---|---|
| `scripts/02_preprocess.py` | Tokenizes injected parquet splits and writes the HuggingFace dataset. |
| `scripts/02_smoke_test.py` | Runs a small DistilBERT smoke test to validate Trainer, metrics, and dataset wiring. |
| `src/pii_masking/day2_preprocessing.py` | Reusable tokenization and Strategy B BIO label-alignment logic. |
| `src/pii_masking/day2_metrics.py` | Reusable seqeval and token-level metric helpers for encoder checks. |
| `data/processed/hf_dataset/` | Tokenized HuggingFace `DatasetDict` consumed by the Kaggle encoder notebook. |
| `models/smoke_test_distilbert/smoke_test_results.json` | Saved local smoke-test metrics. |
| `reports/figures/day2_token_length_distribution.png` | Token-length EDA figure generated during Day 1 validation and reused here to justify `max_length=256`. |


## How To Reproduce Day 2

Run from the project root after Day 1 artifacts exist:

```bash
python scripts/02_preprocess.py
python scripts/02_smoke_test.py
```

Day 2 consumes `data/processed/train_with_emails.parquet`, `data/processed/val_with_emails.parquet`, and `data/processed/test_injected.parquet` from Day 1.


## Token-Length Distribution Choosing max_length=256 for Full Training

WikiNeural sentences are short (word-level median 23 tokens, p95=47), but subword tokenization expands token count significantly a 47-word sentence with a multi-part injected email address can become 60+ DeBERTa SentencePiece tokens. The histogram below was computed over the injected train split after DistilBERT WordPiece tokenization. The p99 falls below 128 subword tokens, so DistilBERT's default 128-token limit is lossless. DeBERTa is given `max_length=256` to accommodate its slower SentencePiece expansion and to ensure injected email tokens near a long sentence's end are never truncated.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

hist_path = Path('../reports/figures/day2_token_length_distribution.png')
if hist_path.exists():
    display(Image(filename=str(hist_path)))
else:
    print(f'Histogram not found at {hist_path}. Run scripts/01_validate_data.py first.')


## HuggingFace DatasetDict Structure Split Verification Before Kaggle Upload

`02_preprocess.py` reads the three injected parquet files and writes a `DatasetDict` with `train`, `validation`, and `test` splits into `data/processed/hf_dataset/`. The test split is included so Day 5 encoder inference can load it from the same dataset artifact but it is never passed to `Trainer.eval_dataset` during training. Confirming all three split directories exist here catches partial preprocessing runs (e.g. only `train` was written) before the dataset is zipped and uploaded to Kaggle, where a missing split would produce a cryptic `KeyError` inside the training notebook rather than a clear missing-file error.

In [ ]:
import json
from pathlib import Path

hf_path = Path('../data/processed/hf_dataset')
dataset_dict_path = hf_path / 'dataset_dict.json'
if dataset_dict_path.exists():
    dataset_meta = json.loads(dataset_dict_path.read_text(encoding='utf-8'))
    print('HF dataset path:', hf_path)
    print(json.dumps(dataset_meta, indent=2))
    for split in ['train', 'validation', 'test']:
        split_path = hf_path / split
        print(f'{split}:', 'exists' if split_path.exists() else 'missing')
else:
    print(f'HuggingFace dataset metadata not found at {dataset_dict_path}. Run scripts/02_preprocess.py first.')


## Smoke-Test Results Pipeline Sanity Before the Kaggle GPU Run

The smoke test runs one DistilBERT epoch on 2 500 training examples at `max_length=64` on CPU. Its purpose is not to produce final metrics but to confirm full pipeline wiring: `DataCollatorForTokenClassification` pads correctly, `-100` sentinel labels are masked before seqeval, `EarlyStoppingCallback` does not error when the best checkpoint is at epoch 1, and `compute_metrics` returns the expected key names for `metric_for_best_model`. A validation F1 of 0.948 at this reduced scale confirms label alignment is correct a misaligned strategy such as labeling all subword continuations with the first-subword tag rather than using `-100` would push F1 below 0.70 at this data scale.

In [ ]:
import json
from pathlib import Path

smoke_path = Path('../models/smoke_test_distilbert/smoke_test_results.json')
if smoke_path.exists():
    smoke = json.loads(smoke_path.read_text(encoding='utf-8'))
    metrics = smoke.get('eval_metrics', {})
    print('Smoke test artifact:', smoke_path)
    print('Model:', smoke.get('model'))
    print('Epochs:', smoke.get('epochs'))
    print('Train subset:', smoke.get('train_subset'))
    print('Validation subset:', smoke.get('val_subset'))
    print('Smoke sequence length:', smoke.get('smoke_seq_len'))
    print('Train loss:', smoke.get('train_loss'))
    print('Validation precision:', metrics.get('eval_overall_precision'))
    print('Validation recall:', metrics.get('eval_overall_recall'))
    print('Validation F1:', metrics.get('eval_overall_f1'))
    print('PER F1:', metrics.get('eval_per_f1'))
    print('EMAIL F1:', metrics.get('eval_email_f1'))
    print('Note:', smoke.get('note'))
else:
    print(f'Smoke test results not found at {smoke_path}. Run scripts/02_smoke_test.py if needed.')


## Notes For Evaluators

- This notebook documents and inspects Day 2 outputs; it does not replace the scripts.
- `scripts/02_preprocess.py` is the source of truth for tokenization and label alignment.
- `scripts/02_smoke_test.py` is the source of truth for the local encoder pipeline smoke test.
- Full encoder training happens later in `notebooks/03_encoder_training.ipynb` on Kaggle/GPU.
